In [ ]:
import time
import json
import re
import pandas as pd
import spacy
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import WebDriverException, InvalidSessionIdException
from webdriver_manager.chrome import ChromeDriverManager

In [ ]:
nlp = spacy.load("de_core_news_md")
OUTPUT = "sz_scrapaped.csv"
SOURCE = "Sueddeutsche Zeitung"
LANGUAGE = "German"
CLIMATE_KEYWORDS = ["klima","klimawandel","klimakrise","erderwärmung", "globale erwärmung","treibhauseffekt","treibhausgas",
    "co2","kohlendioxid","emission","emissionen", "klimaschutz","klimapolitik","klimaneutral","dekarbonisierung",

    "klimaziel","klimaziele","co2-preis","emissionshandel", "klimaschutzgesetz","paris-abkommen","pariser abkommen",
    "eu-klimapaket","green deal","klimaplan","klimabericht","ipcc",

    "energiewende","erneuerbare energien","windkraft", "solarenergie","photovoltaik","wasserstoff",
    "kohlekraft","kohleausstieg","atomkraft", "stromnetz","netzausbau","e-mobilität", "verbrennerverbot","industrieemissionen",

    "hitzewelle","dürre","hochwasser","überschwemmung", "starkregen","waldbrand","extremwetter", "gletscherschmelze","meeresspiegel","wasserknappheit",
    "klimafolgen","artensterben",

    "klimarisiko","nachhaltigkeit","nachhaltige investitionen", "esg","grüne finanzierung","klimafonds", "energiepreise","strompreis","gaspreis",

    "landwirtschaft","trockenheit","ernteschäden", "umweltschutz","biodiversität","naturschutz", "ökosystem","umweltpolitik",

    "fridays for future","klimaaktivisten", "klimaprotest","letzte generation","klimabewegung", "klimadebatte"]
STRICT_KEYWORDS = ["klima","klimakrise","klimawandel","erderwärmung", "co2","kohlendioxid","emission","emissionen",
    "energiewende","klimaschutz","klimaneutral",
    "treibhauseffekt","treibhausgas", "hitzewelle","dürre","hochwasser", "wasserknappheit","starkregen","waldbrand"]
CLASSIFICATION_KEYWORDS = {
    "Climate_Policy": ["gesetz","politik","regierung","beschluss", "verordnung","ziel","klimaziel", "bundestag","eu","parlament","ministerium"],
    "Climate_Science": ["studie","forschung","wissenschaft", "ipcc","daten","analyse","bericht", "modell","forscher"],
    "Energy_Transition": ["energiewende","erneuerbar","solar", "windkraft","kohlekraft","atomkraft", "wasserstoff","stromnetz"],
    "Climate_Economy": ["kosten","industrie","wirtschaft", "markt","unternehmen","investition","preis","arbeitsplätze"],
    "Climate_Activism": ["protest","demonstration","aktivisten","fridays for future","ngo","bewegung"],
    "Climate_Impact": ["hitzewelle","dürre","hochwasser","überschwemmung","starkregen","waldbrand","extremwetter"],
    "Climate_Geopolitics": ["china","usa","eu","russland", "international","global","weltweit", "g7","g20"],
    "Climate_Opinion": ["meinung","kommentar","kolumne", "leitartikel","debatte","gastbeitrag"]}

In [ ]:
def start_browser():
    options = Options()
    options.add_argument("--start-maximized")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()),options=options)
    return driver
driver = start_browser()
wait = WebDriverWait(driver,10)
def clean_text(text):
    if not text:
        return None
    return re.sub(r"\s+"," ",text).strip()
def strict_relevance(title,content):
    t = title.lower()
    c = content.lower()
    lead = " ".join(c.split()[:300])
    bad_context = ["stadion","team","fußball","arbeitsklima"]
    if any(b in t for b in bad_context):
        return False
    return (any(k in t for k in STRICT_KEYWORDS if k!="klima") or sum(k in lead for k in STRICT_KEYWORDS) >= 3)
def analyze_text(text):
    doc = nlp(text)
    actors = sorted(set(ent.text for ent in doc.ents if ent.label_ in ["PER","ORG"]))
    return actors,len(actors),len(list(doc.sents)),len(text)
def classify_article(title,content):
    text = (title + " " + content).lower()
    scores = {}
    for category,keywords in CLASSIFICATION_KEYWORDS.items():
        score = sum(text.count(k) for k in keywords)
        scores[category] = score
    best = max(scores,key=scores.get)
    if scores[best]==0:
        return "Other"
    return best
def accept_cookies():
    try:
        btn = WebDriverWait(driver,3).until(EC.element_to_be_clickable((By.XPATH,"//button[contains(.,'Akzeptieren')]")))
        btn.click()
    except:
        pass
def extract_article(url):
    global driver
    try:
        driver.get(url)
    except (WebDriverException,InvalidSessionIdException):
        try:
            driver.quit()
        except:
            pass
        driver=start_browser()
        driver.get(url)
    accept_cookies()
    time.sleep(3)
    soup=BeautifulSoup(driver.page_source,"html.parser")
    headline=None
    content=None
    author=None
    pub_date=None
    for script in soup.find_all("script",type="application/ld+json"):
        try:
            data=json.loads(script.string)
        except:
            continue
        if isinstance(data,dict) and data.get("@type")=="NewsArticle":
            headline=clean_text(data.get("headline"))
            content=clean_text(data.get("articleBody"))
            pub_date=data.get("datePublished")
            auth=data.get("author")
            if isinstance(auth,dict):
                author=auth.get("name")
            elif isinstance(auth,list) and auth:
                author=auth[0].get("name")
            break
    if not content:
        paragraphs=[p.get_text(strip=True) for p in soup.find_all("p") if len(p.get_text(strip=True))>80]
        content=clean_text(" ".join(paragraphs))
    return headline,content,author,pub_date

In [ ]:
rows=[]
visited=set()

In [ ]:
for keyword in CLIMATE_KEYWORDS:
    print("\nSearching:",keyword)
    try:
        search_url=f"https://www.sueddeutsche.de/suche?search={keyword}"
        driver.get(search_url)
        accept_cookies()
        time.sleep(3)
        soup=BeautifulSoup(driver.page_source,"html.parser")
        links=soup.select("a[href*='sueddeutsche.de']")
    except Exception:
        print("Search failed:",keyword)
        continue
    count = 0
    for link in links:
        if count >= 3:
            break
        url=link.get("href")
        if not url:
            continue
        if not url.startswith("http"):
            url="https://www.sueddeutsche.de"+url
        if url in visited:
            continue
        visited.add(url)
        try:
            headline,content,author,pub_date=extract_article(url)
        except:
            continue
        if not headline or not content:
            continue
        if not strict_relevance(headline,content):
            continue
        actors,actor_count,sent_count,length=analyze_text(content)
        intro = list(nlp(content).sents)[0].text if content else None
        article_class = classify_article(headline,content)
        rows.append({
            "URL": url,
            "Source": SOURCE,
            "Language": LANGUAGE,
            "Published_Date": pub_date,
            "Keyword_Matched": keyword,
            "Article_Classification": article_class,
            "Headline": headline,
            "Intro": intro,
            "Content": content,
            "Content_Length": length,
            "Sentence_Count": sent_count,
            "Actors": ", ".join(actors),
            "Actor_Count": actor_count,
            "Author": author
        })
        print("Saved")
        count += 1
        time.sleep(1)

In [ ]:
df=pd.DataFrame(rows)
df.to_csv(OUTPUT,index=False,encoding="utf-8-sig")
print("Articles collected:",len(df))
driver.quit()